# Area of Applicability

In spatial modelling, sampling of training data requires care, to ensure the model is applicable across the study area. However, even in the best scenarios, some observations in the test or inference sets might just be too dissimilar to the values in the traning set, causing the model to loose its reliability. Area of Applicability, proposed by {cite:t}`meyer2021` estimates whether the model is applicable in a new location or not, based on the feature-weighted distance to its nearest training sample location.

In [ ]:
import geodatasets
import geopandas as gpd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

from spml.metrics import area_of_applicability

Use the data for US counties. This example simulates the situation where we have known values for the Southern states, fit a model based on those and use it to predict homicide rate across the rest of the US.

In [ ]:
ncovr = gpd.read_file(geodatasets.get_path("geoda ncovr")).to_crs(epsg=5070)
ncovr["SOUTH"] = ncovr["SOUTH"].astype(bool)

Check the subset used for training:

In [ ]:
ncovr.plot("SOUTH", categorical=True, legend=True).set_axis_off()

Fit a simple model predicting the homicide rate for 1990 based on a subset of socio-economic indicators for the same year.

In [ ]:
cols = [c for c in ncovr.columns.drop(["HR90", "HC90"]) if "90" in c]

south = ncovr.loc[ncovr.SOUTH]
model = LinearRegression().fit(south[cols], south["HR90"])

Check the performance:

In [ ]:
print(f"Train score: {model.score(south[cols], south['HR90'])}")
print(
    f"Test score: {model.score(ncovr[~ncovr.SOUTH][cols], ncovr[~ncovr.SOUTH]['HR90'])}"
)

It is clear that there are clearly some issues with the model generalisability outside of the Southern states. If you would do the prediction across the entire dataset, you would get a map like this.

In [ ]:
pred = model.predict(ncovr[cols])
ncovr.plot(
    pred, legend=True, scheme="naturalbreaks", legend_kwds={"colorbar": True}
).set_axis_off()

However, it is likely that not all areas can be trusted. Estimate the area of applicability, which returns a boolean mask capturing locations where the model is applicable and those where it is not.

In [ ]:
aoa = area_of_applicability(
    ncovr[cols], south[cols], y_train=south["HR90"], model=model
)
aoa

On a map, the mask looks like the left panel in the figure below. The right panel then reports the prediction masked-out -- ignoring locations where the model is not applicable.

In [ ]:
f, axs = plt.subplots(1, 2, figsize=(12, 5))
ncovr.plot(aoa, legend=True, ax=axs[0])
ncovr[aoa].plot(pred[aoa], ax=axs[1])
axs[0].set_title("Area of Applicability")
axs[1].set_title("AoA-masked prediction")
for ax in axs:
    ax.set_axis_off()

## Diagnostics

The area of applicability is derived from the Dissimilarity Index (DI) and Local Point Density (LPD). It computes DI, identifies a threshold value (by default usign Tukey rule -- 75th percentile + 1.5 * IQR) and returns a boolean mask reflecting DI below or above the threshold. All these components can be accessed through the diagnostic run of `area_of_applicability`.

In [ ]:
aoa_diag = area_of_applicability(
    ncovr[cols],
    south[cols],
    y_train=south["HR90"],
    model=model,
    return_diagnostics=True,
)
aoa_diag

You can therefore visualise DI or LPD directly.

In [ ]:
f, axs = plt.subplots(1, 2, figsize=(12, 5))

ncovr.plot(aoa_diag.dissimilarity_index, vmax=aoa_diag.cutpoint, legend=True, ax=axs[0])
ncovr.plot(aoa_diag.lpd, legend=True, ax=axs[1])
axs[0].set_title("Dissimilarity Index")
axs[1].set_title("Local Point Density")
for ax in axs:
    ax.set_axis_off()

## Options

By default, the function computes the feature weights using permutation importance to give more important features higher weight. This can be relatively costly, so you can treat all weights uniformly (or pass pre-defined array of weights). Similarly, you can change the method of the threshold derivation (or pass a custom float).

In [ ]:
aoa_uniform = area_of_applicability(
    ncovr[cols], south[cols], feature_weights="uniform", threshold="mad"
)

This results in a slightly different AoA mask.

In [ ]:
f, axs = plt.subplots(1, 2, figsize=(12, 5))

ncovr.plot(aoa_uniform, legend=True, ax=axs[0])
ncovr.plot(
    aoa == aoa_uniform,
    legend=True,
    ax=axs[1],
    cmap={True: "steelblue", False: "salmon"},
)
axs[0].set_title("AoA (uniform weights, MAD)")
axs[1].set_title("Agreement bewteen methods")
for ax in axs:
    ax.set_axis_off()